# Оценка модели spell correction (stage 2) на всех датасетах
Асинхронный запуск запросов, подсчёт метрик, итоговая таблица.

Используется модель, обученная в две стадии: сперва на датасете `synth_spell_correction_1m`, а затем на `spell_correction_30k`

In [ ]:
import asyncio

import nest_asyncio
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
from openai import AsyncOpenAI
from sage.utils import load_available_dataset_from_hf, DatasetsAvailable
from sage.evaluation import Scorer

# В Jupyter уже работает event loop, поэтому нужен nest_asyncio
nest_asyncio.apply()

In [ ]:
# Настройки
BASE_URL = "http://localhost:9900/v1"
API_KEY = "1234"
MODEL = "spell_correction_stage_2"
TEMPERATURE = 0.1

# Максимум одновременных запросов
MAX_CONCURRENT = 512

PROMPT_TEMPLATE = (
    "Исходный текст:\n{text}\n\nОтредактируй исходный текст, исправив ошибки.\n"
)

client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

In [ ]:
async def correct_text(sem: asyncio.Semaphore, text: str) -> str:
    """Отправляет один запрос к модели с ограничением параллелизма."""
    async with sem:
        response = await client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}],
            temperature=TEMPERATURE,
        )
    return response.choices[0].message.content


async def run_corrections(sources: list[str]) -> list[str]:
    """Запускает все запросы асинхронно с progress bar."""
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    tasks = [correct_text(sem, s) for s in sources]
    return await tqdm_asyncio.gather(*tasks, desc="Inferencing")

In [ ]:
def load_dataset(name: str):
    """
    Возвращает (sources, corrections) для любого датасета.
    RUSpellRU и MultidomainGold возвращают два списка напрямую,
    остальные — DataFrame с колонками source/correction.
    """
    if (
        name == DatasetsAvailable.RUSpellRU.name
        or name == DatasetsAvailable.MultidomainGold.name
    ):
        sources, corrections = load_available_dataset_from_hf(
            name, for_labeler=True, split="test"
        )
        return list(sources), list(corrections)
    else:
        df = load_available_dataset_from_hf(name, for_labeler=False)
        return df["source"].tolist(), df["correction"].tolist()

In [ ]:
DATASET_NAMES = [d.name for d in DatasetsAvailable]
print("Датасеты:", DATASET_NAMES)

In [ ]:
DATASET_NAMES = [
    "MultidomainGold",
    "RUSpellRU",
    "MedSpellchecker",
    "GitHubTypoCorpusRu",
]

In [ ]:
scorer = Scorer()
all_results = {}

for ds_name in DATASET_NAMES:
    print(f"\n{'=' * 60}")
    print(f"Датасет: {ds_name}")

    sources, corrections = load_dataset(ds_name)
    print(f"Примеров: {len(sources)}")

    # Запускаем асинхронный инференс
    predictions = asyncio.run(run_corrections(sources))

    # Считаем метрики
    metrics = scorer.score(
        sources,
        corrections,
        predictions,
        metrics=["ruspelleval", "errant"],
    )
    all_results[ds_name] = metrics
    print("Метрики:", metrics)

print("\nГотово!")

In [ ]:
# Итоговая таблица
df_results = pd.DataFrame(all_results).T
df_results.index.name = "Dataset"

# Округляем до 2 знаков
df_results = df_results.round(2)

# Группируем колонки для читаемости
base_cols = ["Precision", "Recall", "F1"]
case_cols = ["CASE_Precision", "CASE_Recall", "CASE_F1"]
spell_cols = ["SPELL_Precision", "SPELL_Recall", "SPELL_F1"]
punct_cols = ["PUNCT_Precision", "PUNCT_Recall", "PUNCT_F1"]
yo_cols = ["YO_Precision", "YO_Recall", "YO_F1"]

ordered_cols = base_cols + case_cols + spell_cols + punct_cols + yo_cols
# Оставляем только те, что реально пришли от scorer
ordered_cols = [c for c in ordered_cols if c in df_results.columns]

df_display = df_results[ordered_cols]

In [ ]:
df_display

In [ ]:
df_display.to_csv("../data/metrics/spell_correction_stage_2.csv", index=False)